# 🗺️ Notebook 06 — Segmentação Geográfica de Clientes

Este notebook adiciona a **dimensão geográfica** ao projeto de segmentação, respondendo à pergunta:

> *Onde estão os nossos clientes e quais regiões concentram mais potencial?*

---

### Estrutura
| Seção | Ferramenta | Objetivo |
|:---|:---|:---|
| 1 | Folium | Mapa interativo com todos os clientes |
| 2 | DBSCAN + Folium | Clustering geográfico sem número fixo de grupos |
| 3 | K-Means + Plotly | Clustering com regiões definidas + mapa Plotly |
| 4 | Plotly Charts | Distribuição por estado, cidade e cluster |

---

## ⚙️ 0. Imports e Configuração

In [1]:
import pandas as pd
import numpy as np
import folium
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

print('Bibliotecas carregadas com sucesso.')

Bibliotecas carregadas com sucesso.


## 📂 1. Carregamento dos Dados

In [2]:
df = pd.read_csv('../data/df_clientes.csv')
df = df.dropna(subset=['latitude', 'longitude'])
df = df[['cliente_id','nome', 'estado', 'cidade', 'cep', 'latitude', 'longitude']]
df = df[df['estado'] == 'GO']

print(f'Total de clientes com coordenadas: {len(df)}')
print(f'Colunas disponíveis: {list(df.columns)}')
df.head(n=1)

Total de clientes com coordenadas: 61
Colunas disponíveis: ['cliente_id', 'nome', 'estado', 'cidade', 'cep', 'latitude', 'longitude']


,cliente_id,nome,estado,cidade,cep,latitude,longitude
14,CUST-1014,Ana Silva,GO,Goiânia,74884-010,-16.740249,-49.256078


In [3]:
# Ponto central do mapa (média das coordenadas)
center_lat = df['latitude'].mean()
center_lon = df['longitude'].mean()

print(f'Centro geográfico da base: ({center_lat:.4f}, {center_lon:.4f})')
print(f'Estados representados: {sorted(df["estado"].unique())}')
print(f'Cidades únicas: {df["cidade"].nunique()}')

Centro geográfico da base: (-16.8541, -49.0049)
Estados representados: ['GO']
Cidades únicas: 2


---
## 🗺️ Seção A — Mapa Interativo com Folium

Visualização de todos os clientes como pontos no mapa.  
Passe o mouse sobre um ponto para ver nome, cidade e estado.

### A.1 — Todos os Clientes (pontos simples)

In [4]:
mapa_base = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=5,
    tiles='CartoDB positron'
)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color='#2563EB',
        fill=True,
        fill_color='#2563EB',
        fill_opacity=0.7,
        tooltip=(
            f"<b>{row['nome']}</b><br>"
            f"{row['cidade']} — {row['estado']}<br>"
            f"CEP: {row['cep']}"
        )
    ).add_to(mapa_base)

mapa_base

A. 2 — Clientes por Por Cidade (círculos proporcionais)

In [5]:
df_estatdo_go = df[df['estado'] == 'GO']
print(f'Total de clientes em GO: {len(df_estatdo_go)}')

Total de clientes em GO: 61


In [6]:
mapa_go = folium.Map(
    location=[df_estatdo_go['latitude'].mean(), df_estatdo_go['longitude'].mean()],
    zoom_start=6,
    tiles='CartoDB positron'
)

for _, row in df_estatdo_go.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color='#2563EB',
        fill=True,
        fill_color='#2563EB',
        fill_opacity=0.7,
        tooltip=(
            f"<b>{row['nome']}</b><br>"
            f"{row['cidade']} — {row['estado']}<br>"
            f"CEP: {row['cep']}"
        )
    ).add_to(mapa_go)
    
mapa_go

### A.3 — Segmentação por Estado e Cidade

Filtre a base por **estado** e, opcionalmente, por **cidade** para visualizar um recorte geográfico específico.  
Ajuste as variáveis `ESTADO` e `CIDADE` (use `None` para exibir todo o estado).

In [7]:
# ── Configuração ────────────────────────────────────────────────────────────
ESTADO = 'GO'       # sigla do estado (obrigatório)
CIDADE = 'Goiânia'  # nome da cidade (ou None para todo o estado)
# ────────────────────────────────────────────────────────────────────────────

df_filtro = df[df['estado'] == ESTADO]
if CIDADE:
    df_filtro = df_filtro[df_filtro['cidade'] == CIDADE]

if df_filtro.empty:
    print(f'Nenhum cliente encontrado para estado={ESTADO}, cidade={CIDADE}.')
else:
    titulo = f'{CIDADE} / {ESTADO}' if CIDADE else ESTADO
    print(f'Clientes encontrados — {titulo}: {len(df_filtro)}')
    print(f'Cidades no recorte: {sorted(df_filtro["cidade"].unique())}')
    print(df_filtro[['cliente_id', 'nome', 'cidade', 'estado']].head())

Clientes encontrados — Goiânia / GO: 58
Cidades no recorte: ['Goiânia']
   cliente_id            nome   cidade estado
14  CUST-1014       Ana Silva  Goiânia     GO
16  CUST-1016   João Ferreira  Goiânia     GO
33  CUST-1033    Gisele Souza  Goiânia     GO
39  CUST-1039  João Rodrigues  Goiânia     GO
42  CUST-1042      Hugo Souza  Goiânia     GO


In [8]:
mapa_filtro = folium.Map(
    location=[df_filtro['latitude'].mean(), df_filtro['longitude'].mean()],
    zoom_start=11 if CIDADE else 6,
    tiles='CartoDB positron'
)

for _, row in df_filtro.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color='#2563EB',
        fill=True,
        fill_color='#2563EB',
        fill_opacity=0.7,
        tooltip=(
            f"<b>{row['nome']}</b><br>"
            f"{row['cidade']} — {row['estado']}<br>"
            f"CEP: {row['cep']}"
        )
    ).add_to(mapa_filtro)

titulo = f'{CIDADE} / {ESTADO}' if CIDADE else ESTADO
folium.map.Marker(
    [df_filtro['latitude'].mean(), df_filtro['longitude'].mean()],
    icon=folium.DivIcon(
        html=f'<div style="font-size:13px;font-weight:bold;color:#1e3a5f;">'
             f'{titulo} — {len(df_filtro)} clientes</div>'
    )
).add_to(mapa_filtro)

mapa_filtro

---
## 🔵 Seção B — Clustering Geográfico com DBSCAN + Folium

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) é ideal para dados geográficos porque:
- Não exige definir o número de clusters previamente
- Detecta clusters de formato irregular (ex: cidades ao longo de uma rodovia)
- Marca clientes isolados como **ruído** em vez de forçá-los em um grupo artificial

Usamos a métrica **Haversine** que calcula distância real na superfície da Terra.

### B.1 — Aplicando DBSCAN

In [9]:
coords_rad = np.radians(df[['latitude', 'longitude']].values)

# eps = raio em radianos (50 km / raio da Terra)
# Aumente eps para clusters maiores, diminua para mais granularidade
RAIO_KM = 50
eps_rad = RAIO_KM / 6371.0

dbscan = DBSCAN(
    eps=eps_rad,
    min_samples=3,
    algorithm='ball_tree',
    metric='haversine'
)

df['cluster_dbscan'] = dbscan.fit_predict(coords_rad)

n_clusters = len(set(df['cluster_dbscan'])) - (1 if -1 in df['cluster_dbscan'].values else 0)
n_ruido = (df['cluster_dbscan'] == -1).sum()

print(f'Clusters encontrados: {n_clusters}')
print(f'Clientes isolados (ruído): {n_ruido} ({n_ruido/len(df)*100:.1f}%)')
print(f'\nDistribuição por cluster:')
print(df['cluster_dbscan'].value_counts().sort_index())

Clusters encontrados: 2
Clientes isolados (ruído): 0 (0.0%)

Distribuição por cluster:
cluster_dbscan
0    58
1     3
Name: count, dtype: int64


### B.2 — Mapa DBSCAN (clusters coloridos)

In [10]:
PALETA = [
    '#E63946', '#2A9D8F', '#E9C46A', '#F4A261', '#264653',
    '#8338EC', '#FB5607', '#3A86FF', '#06D6A0', '#FFB703'
]

def cor_dbscan(cluster_id):
    if cluster_id == -1:
        return '#AAAAAA'  # cinza = ruído
    return PALETA[cluster_id % len(PALETA)]

mapa_dbscan = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=5,
    tiles='CartoDB positron'
)

for _, row in df.iterrows():
    cluster_id = row['cluster_dbscan']
    cor = cor_dbscan(cluster_id)
    label = f'Cluster {cluster_id}' if cluster_id != -1 else 'Isolado'

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.85,
        tooltip=(
            f"<b>{row['nome']}</b><br>"
            f"{row['cidade']} — {row['estado']}<br>"
            f"<i style='color:{cor}'>{label}</i>"
        )
    ).add_to(mapa_dbscan)

mapa_dbscan

---
## 🟠 Seção C — Clustering com K-Means + Mapa Plotly

**K-Means** divide os clientes em **K regiões de igual influência**.  
Útil quando você quer definir explicitamente quantas zonas de atuação existem (ex: 4 regiões comerciais).

Usamos o **Método do Cotovelo** para escolher o K ideal.

### C.1 — Método do Cotovelo

In [11]:
X = df[['latitude', 'longitude']].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inercias = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)

fig_cotovelo = px.line(
    x=list(K_range),
    y=inercias,
    markers=True,
    title='Método do Cotovelo — K-Means Geográfico',
    labels={'x': 'Número de Clusters (K)', 'y': 'Inércia'},
    template='plotly_white'
)
fig_cotovelo.update_traces(line_color='#2563EB', marker_color='#E63946', marker_size=8)
fig_cotovelo.show()

### C.2 — Aplicando K-Means

> **Ajuste `K_IDEAL`** com base no cotovelo identificado no gráfico acima.

In [12]:
K_IDEAL = 3  # <- ajuste após analisar o cotovelo

kmeans = KMeans(n_clusters=K_IDEAL, random_state=42, n_init=10)
df['cluster_kmeans'] = kmeans.fit_predict(X_scaled)
df['regiao'] = df['cluster_kmeans'].apply(lambda x: f'Região {x + 1}')

print(f'K-Means aplicado com K={K_IDEAL}')
print(df['regiao'].value_counts())

K-Means aplicado com K=3
regiao
Região 1    45
Região 3    13
Região 2     3
Name: count, dtype: int64


### C.3 — Mapa Plotly (scatter_mapbox)

In [13]:
fig_mapa_kmeans = px.scatter_mapbox(
    df,
    lat='latitude',
    lon='longitude',
    color='regiao',
    hover_name='nome',
    hover_data={
        'cidade': True,
        'estado': True,
        'cep': False,
        'latitude': False,
        'longitude': False
    },
    zoom=4,
    center={'lat': center_lat, 'lon': center_lon},
    title=f'K-Means Geográfico — {K_IDEAL} Regiões',
    mapbox_style='carto-positron',
    height=600,
    template='plotly_white'
)
fig_mapa_kmeans.update_layout(legend_title_text='Região')
fig_mapa_kmeans.show()

---
## 📊 Seção D — Análises Complementares

Gráficos que respondem: *onde estão concentrados nossos clientes?*

### D.1 — Clientes por Estado

In [14]:
clientes_estado = (
    df.groupby('estado')
    .size()
    .reset_index(name='total')
    .sort_values('total', ascending=False)
)

fig_estado = px.bar(
    clientes_estado,
    x='estado',
    y='total',
    title='Distribuição de Clientes por Estado',
    color='total',
    color_continuous_scale='Blues',
    labels={'estado': 'Estado', 'total': 'Nº de Clientes'},
    text='total',
    template='plotly_white'
)
fig_estado.update_traces(textposition='outside')
fig_estado.update_layout(coloraxis_showscale=False)
fig_estado.show()

### D.2 — Top 15 Cidades

In [15]:
top_cidades = (
    df.groupby(['cidade', 'estado'])
    .size()
    .reset_index(name='total')
    .sort_values('total', ascending=False)
    .head(15)
)
top_cidades['cidade_uf'] = top_cidades['cidade'] + ' / ' + top_cidades['estado']

fig_cidades = px.bar(
    top_cidades,
    x='total',
    y='cidade_uf',
    orientation='h',
    title='Top 15 Cidades com Mais Clientes',
    color='total',
    color_continuous_scale='Teal',
    labels={'cidade_uf': 'Cidade / UF', 'total': 'Nº de Clientes'},
    text='total',
    height=520,
    template='plotly_white'
)
fig_cidades.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False
)
fig_cidades.update_traces(textposition='outside')
fig_cidades.show()

### D.3 — Distribuição por Cluster DBSCAN

In [16]:
cluster_summary = (
    df.groupby('cluster_dbscan')
    .agg(
        clientes=('cliente_id', 'count'),
        cidades=('cidade', 'nunique'),
        estados=('estado', 'nunique')
    )
    .reset_index()
)
cluster_summary['label'] = cluster_summary['cluster_dbscan'].apply(
    lambda x: 'Isolados' if x == -1 else f'Cluster {x}'
)

fig_dist = px.bar(
    cluster_summary,
    x='label',
    y='clientes',
    title='Clientes por Cluster DBSCAN',
    color='clientes',
    color_continuous_scale='Viridis',
    text='clientes',
    labels={'label': 'Cluster', 'clientes': 'Nº de Clientes'},
    template='plotly_white'
)
fig_dist.update_traces(textposition='outside')
fig_dist.update_layout(coloraxis_showscale=False)
fig_dist.show()

print('\nResumo detalhado:')
print(cluster_summary[['label', 'clientes', 'cidades', 'estados']].to_string(index=False))


Resumo detalhado:
    label  clientes  cidades  estados
Cluster 0        58        1        1
Cluster 1         3        1        1


### D.4 — Distribuição por Região K-Means

In [17]:
regiao_summary = (
    df.groupby('regiao')
    .agg(
        clientes=('cliente_id', 'count'),
        cidades=('cidade', 'nunique'),
        estados=('estado', 'nunique')
    )
    .reset_index()
    .sort_values('clientes', ascending=False)
)

fig_regiao = px.pie(
    regiao_summary,
    names='regiao',
    values='clientes',
    title=f'Participação por Região K-Means (K={K_IDEAL})',
    hole=0.4,
    template='plotly_white'
)
fig_regiao.update_traces(textinfo='label+percent+value')
fig_regiao.show()

print('\nResumo por Região:')
print(regiao_summary.to_string(index=False))


Resumo por Região:
  regiao  clientes  cidades  estados
Região 1        45        1        1
Região 3        13        1        1
Região 2         3        1        1


---
## 💾 Exportar Resultado

In [18]:
colunas_export = ['cliente_id', 'nome', 'cidade', 'estado', 'cep',
                  'latitude', 'longitude', 'cluster_dbscan', 'regiao']

df_geo = df[colunas_export].copy()
df_geo.to_csv('../data/df_clientes_geo.csv', index=False)

print(f'Arquivo exportado: df_clientes_geo.csv ({len(df_geo)} registros)')
df_geo.head()

Arquivo exportado: df_clientes_geo.csv (61 registros)


,cliente_id,nome,cidade,estado,cep,latitude,longitude,cluster_dbscan,regiao
14,CUST-1014,Ana Silva,Goiânia,GO,74884-010,-16.740249,-49.256078,0,Região 3
16,CUST-1016,João Ferreira,Goiânia,GO,74040-070,-16.685586,-49.264859,0,Região 1
33,CUST-1033,Gisele Souza,Goiânia,GO,74825-440,-16.706728,-49.260820,0,Região 1
39,CUST-1039,João Rodrigues,Goiânia,GO,74680-030,-16.717985,-49.248071,0,Região 3
42,CUST-1042,Hugo Souza,Goiânia,GO,74680-030,-16.718176,-49.248656,0,Região 3


---
## 📌 Conclusões

| Abordagem | Vantagem | Quando usar |
|:---|:---|:---|
| **DBSCAN** | Detecta clusters naturais, trata outliers | Exploração inicial, sem hipótese de número de zonas |
| **K-Means** | Regiões balanceadas e previsíveis | Quando o negócio precisa de N zonas fixas (ex: 4 regiões comerciais) |

### Próximos Passos
- Cruzar `cluster_dbscan` / `regiao` com os clusters RFM existentes (`Campeões`, `Potenciais`, `Hibernando`)
- Identificar regiões com alto potencial mas baixo engajamento atual
- Priorizar ações de expansão nas zonas de maior concentração de clientes de alto valor